In [16]:
import sqlite3
import pandas as pd

users = pd.read_csv("users.csv")

conn = sqlite3.connect("enterprise_database.db")

users.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql(
    "SELECT * FROM users",conn
)

,user_id,name,status,updated_at
0,101,Vikas,Active,2025-05-13
1,102,Dev,Active,2025-01-02
2,103,Aakash,Inactive,2025-06-19
3,104,Priya,Active,2025-06-19
4,105,Amit,Active,2025-05-05
...,...,...,...,...
95,196,Nisha,Inactive,2025-04-24
96,197,Aakash,Active,2025-06-05
97,198,Priya,Inactive,2025-06-02
98,199,Aakash,Active,2025-04-17


In [17]:
#Data Loading:
import pandas as pd

transactions = pd.read_csv(
    "transactions.csv"
)

exchange_rates = pd.read_csv(
    "exchange_rates.csv"
)

print(transactions)
print(exchange_rates)

     transaction_id  user_id   amount transaction_date currency
0          TXN00001      117  4279.84       2025-01-26      EUR
1          TXN00002      126  1716.44       2025-02-03      EUR
2          TXN00003      128  3639.62       2025-06-24      EUR
3          TXN00004      129    85.50       2025-04-09      EUR
4          TXN00005      191  4160.22       2025-05-21      EUR
...             ...      ...      ...              ...      ...
1495       TXN01496      182  1495.24       2025-03-28      EUR
1496       TXN01497      191  1575.01       2025-01-03      EUR
1497       TXN01498      133   444.86       2025-01-30      EUR
1498       TXN01499      156  3682.71       2025-06-18      EUR
1499       TXN01500      180  1856.34       2025-02-08      EUR

[1500 rows x 5 columns]
           Date  exchange_rate
0    2025-01-01          83.68
1    2025-01-02          83.25
2    2025-01-03          83.71
3    2025-01-06          84.52
4    2025-01-07          84.17
..          ...      

In [18]:
transactions.isnull().sum()

transaction_id      0
user_id             0
amount              0
transaction_date    0
currency            0
dtype: int64

In [19]:
transactions.duplicated().sum()

np.int64(0)

In [20]:
transactions.drop_duplicates(
    inplace=True
)

In [21]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

exchange_rates["Date"] = pd.to_datetime(
    exchange_rates["Date"]
)

In [22]:
query = """
SELECT *
FROM users
"""

users_df = pd.read_sql(
    query,
    conn
)

print(users_df)

    user_id    name    status  updated_at
0       101   Vikas    Active  2025-05-13
1       102     Dev    Active  2025-01-02
2       103  Aakash  Inactive  2025-06-19
3       104   Priya    Active  2025-06-19
4       105    Amit    Active  2025-05-05
..      ...     ...       ...         ...
95      196   Nisha  Inactive  2025-04-24
96      197  Aakash    Active  2025-06-05
97      198   Priya  Inactive  2025-06-02
98      199  Aakash    Active  2025-04-17
99      200   Karan  Inactive  2025-03-01

[100 rows x 4 columns]


In [24]:

merged = pd.merge(
    transactions,
    users_df,
    on="user_id",
    how="inner"
)

merged.head()

,transaction_id,user_id,amount,transaction_date,currency,name,status,updated_at
0,TXN00001,117,4279.84,2025-01-26,EUR,Nisha,Inactive,2025-05-25
1,TXN00002,126,1716.44,2025-02-03,EUR,Nisha,Inactive,2025-04-28
2,TXN00003,128,3639.62,2025-06-24,EUR,Vikas,Inactive,2025-01-11
3,TXN00004,129,85.50,2025-04-09,EUR,Pooja,Inactive,2025-01-28
4,TXN00005,191,4160.22,2025-05-21,EUR,Rohan,Inactive,2025-02-15


In [26]:
merged = pd.merge(
    merged,
    exchange_rates,
    left_on="transaction_date",
    right_on="Date",
    how="left"
)

merged.head()

,transaction_id,user_id,amount,transaction_date,currency,name,status,updated_at,Date_x,exchange_rate_x,Date_y,exchange_rate_y
0,TXN00001,117,4279.84,2025-01-26,EUR,Nisha,Inactive,2025-05-25,NaT,NaN,NaT,NaN
1,TXN00002,126,1716.44,2025-02-03,EUR,Nisha,Inactive,2025-04-28,2025-02-03,83.93,2025-02-03,83.93
2,TXN00003,128,3639.62,2025-06-24,EUR,Vikas,Inactive,2025-01-11,2025-06-24,83.32,2025-06-24,83.32
3,TXN00004,129,85.50,2025-04-09,EUR,Pooja,Inactive,2025-01-28,2025-04-09,84.97,2025-04-09,84.97
4,TXN00005,191,4160.22,2025-05-21,EUR,Rohan,Inactive,2025-02-15,2025-05-21,84.58,2025-05-21,84.58


In [30]:
merged.columns.tolist()


['transaction_id',
 'user_id',
 'amount',
 'transaction_date',
 'currency',
 'name',
 'status',
 'updated_at',
 'Date_x',
 'exchange_rate_x',
 'Date_y',
 'exchange_rate_y']

In [31]:
merged[['exchange_rate_x', 'exchange_rate_y']].head(10)

,exchange_rate_x,exchange_rate_y
0,NaN,NaN
1,83.93,83.93
2,83.32,83.32
3,84.97,84.97
4,84.58,84.58
5,83.79,83.79
6,84.96,84.96
7,83.51,83.51
8,83.52,83.52
9,83.79,83.79


In [37]:
merged.drop(columns=['exchange_rate_x'], inplace=True)

merged.rename(
    columns={'exchange_rate_y': 'exchange_rate'},
    inplace=True
)

In [39]:
merged["amount_usd"] = (
    merged["amount"] /
    merged["exchange_rate"]
)

merged['amount_usd'].head()

0          NaN
1    20.450852
2    43.682429
3     1.006237
4    49.186805
Name: amount_usd, dtype: float64

In [41]:
missing_users = transactions[
    ~transactions["user_id"].isin(
        users_df["user_id"]
    )
]

print(missing_users)

Empty DataFrame
Columns: [transaction_id, user_id, amount, transaction_date, currency]
Index: []


In [44]:
missing_rates = merged[
    merged["exchange_rate"].isna()
]

missing_rates.head()

,transaction_id,user_id,amount,transaction_date,currency,name,status,updated_at,Date_x,Date_y,Date,exchange_rate,amount_usd
0,TXN00001,117,4279.84,2025-01-26,EUR,Nisha,Inactive,2025-05-25,NaT,NaT,NaT,NaN,NaN
11,TXN00012,140,211.96,2025-06-22,EUR,Amit,Active,2025-04-18,NaT,NaT,NaT,NaN,NaN
13,TXN00014,121,2757.65,2025-06-14,EUR,Manav,Active,2025-06-20,NaT,NaT,NaT,NaN,NaN
17,TXN00018,143,3026.61,2025-02-23,EUR,Meera,Inactive,2025-05-22,NaT,NaT,NaT,NaN,NaN
20,TXN00021,152,3264.50,2025-05-03,EUR,Vikas,Inactive,2025-03-17,NaT,NaT,NaT,NaN,NaN


In [46]:
duplicates = transactions[
    transactions.duplicated()
]

print(duplicates)

Empty DataFrame
Columns: [transaction_id, user_id, amount, transaction_date, currency]
Index: []


In [49]:
total_revenue = merged["amount_usd"].sum()

total_transactions = len(merged)

active_users = merged["user_id"].nunique()

avg_transaction = merged[
    "amount_usd"
].mean()

print(total_revenue)
print(total_transactions)
print(active_users)
print(avg_transaction)

31075.823165979233
1500
100
29.90935819632265


In [50]:
merged.to_csv(
    "final_audit_dataset.csv",
    index=False
)